In [1]:
import torch

def box_xyxy_to_xywh(b):
    x1,y1,x2,y2 = b.unbind(-1)
    w = x2 - x1; h = y2 - y1
    cx = x1 + 0.5*w; cy = y1 + 0.5*h
    return torch.stack([cx,cy,w,h], dim=-1)

def box_xywh_to_xyxy(b):
    cx,cy,w,h = b.unbind(-1)
    x1 = cx - 0.5*w; y1 = cy - 0.5*h
    x2 = cx + 0.5*w; y2 = cy + 0.5*h
    return torch.stack([x1,y1,x2,y2], dim=-1)

def iou(boxes1, boxes2):
    # boxes*: [N,4] (xyxy)
    tl = torch.max(boxes1[:,None,:2], boxes2[:,:2])    # [N,M,2]
    br = torch.min(boxes1[:,None,2:], boxes2[:,2:])
    wh = (br - tl).clamp(min=0)
    inter = wh[...,0]*wh[...,1]
    area1 = (boxes1[:,2]-boxes1[:,0])*(boxes1[:,3]-boxes1[:,1])
    area2 = (boxes2[:,2]-boxes2[:,0])*(boxes2[:,3]-boxes2[:,1])
    union = area1[:,None] + area2 - inter
    return inter / (union + 1e-7)

# SSD encode/decode (theo tỉ lệ offsets và log-scale)
def ssd_encode(gt_xyxy, anchors_xywh, variances=(0.1,0.2)):
    gt_xywh = box_xyxy_to_xywh(gt_xyxy)
    g_cxcy = (gt_xywh[:,:2] - anchors_xywh[:,:2]) / anchors_xywh[:,2:]
    g_wh   = torch.log(gt_xywh[:,2:] / anchors_xywh[:,2:])
    v1,v2 = variances
    return torch.cat([(g_cxcy/v1), (g_wh/v2)], dim=-1)

def ssd_decode(pred_t, anchors_xywh, variances=(0.1,0.2)):
    v1,v2 = variances
    cxcy = pred_t[:,:2]*v1*anchors_xywh[:,2:] + anchors_xywh[:,:2]
    wh   = torch.exp(pred_t[:,2:]*v2)*anchors_xywh[:,2:]
    return box_xywh_to_xyxy(torch.cat([cxcy,wh], dim=-1))

In [2]:
import torch

def nms(boxes, scores, iou_th=0.5):
    keep = []
    idxs = scores.argsort(descending=True)
    while idxs.numel() > 0:
        i = idxs[0]; keep.append(i.item())
        if idxs.numel() == 1: break
        iou_vals = iou(boxes[i].unsqueeze(0), boxes[idxs[1:]]).squeeze(0)
        idxs = idxs[1:][iou_vals <= iou_th]
    return keep

# Soft-NMS (Gaussian)
def soft_nms(boxes, scores, iou_th=0.5, sigma=0.5, score_th=0.001):
    idxs = scores.argsort(descending=True)
    boxes = boxes.clone(); scores = scores.clone()
    keep = []
    while idxs.numel() > 0:
        i = idxs[0]; keep.append(i.item())
        if idxs.numel() == 1: break
        others = idxs[1:]
        ov = iou(boxes[i].unsqueeze(0), boxes[others]).squeeze(0)
        weights = torch.exp(- (ov**2) / sigma)
        scores[others] = scores[others]*weights
        idxs = torch.cat([others[scores[others] > score_th]])
        idxs = idxs[scores[idxs].argsort(descending=True)]
    return keep


In [3]:
import torch
def ssd_anchors(feat_h, feat_w, img_h, img_w, scales=[0.1,0.2,0.35], ratios=[1.0,2.0,0.5]):
    # Trả về anchors ở dạng (cx,cy,w,h) tỷ lệ theo pixel
    stride_y = img_h/feat_h; stride_x = img_w/feat_w
    anchors=[]
    for y in range(feat_h):
        for x in range(feat_w):
            cy = (y+0.5)*stride_y; cx=(x+0.5)*stride_x
            for s in scales:
                for r in ratios:
                    w = img_w*s*(r**0.5); h = img_h*s/(r**0.5)
                    anchors.append([cx,cy,w,h])
    return torch.tensor(anchors, dtype=torch.float32)

def yolo_grid(feat_h, feat_w, stride):
    # grid (cx, cy) tính theo pixel (YOLO head dự đoán offset trên grid)
    yv, xv = torch.meshgrid(torch.arange(feat_h), torch.arange(feat_w), indexing='ij')
    grid = torch.stack((xv, yv), 2).view(-1,2).float()
    return (grid + 0.5) * stride  # tâm ô lưới


In [4]:
import torch.nn as nn

class SimpleFPN(nn.Module):
    def __init__(self, c3, c4, c5, out_channels=256):
        super().__init__()
        self.lateral3 = nn.Conv2d(c3, out_channels, 1)
        self.lateral4 = nn.Conv2d(c4, out_channels, 1)
        self.lateral5 = nn.Conv2d(c5, out_channels, 1)
        self.smooth3  = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.smooth4  = nn.Conv2d(out_channels, out_channels, 3, padding=1)

    def forward(self, C3, C4, C5):
        P5 = self.lateral5(C5)
        P4 = self.lateral4(C4) + nn.functional.interpolate(P5, scale_factor=2, mode='nearest')
        P3 = self.lateral3(C3) + nn.functional.interpolate(P4, scale_factor=2, mode='nearest')
        P4 = self.smooth4(P4); P3 = self.smooth3(P3)
        return P3, P4, P5  # có thể tạo thêm P6,P7 bằng stride conv


In [5]:
class SSDHead(nn.Module):
    def __init__(self, in_channels, num_anchors, num_classes):
        super().__init__()
        self.cls = nn.Conv2d(in_channels, num_anchors*num_classes, 3, padding=1)
        self.reg = nn.Conv2d(in_channels, num_anchors*4, 3, padding=1)

    def forward(self, feats):  # feats: list[P3,P4,P5...]
        cls_outs, reg_outs = [], []
        for f in feats:
            cls_outs.append(self.cls(f).permute(0,2,3,1).contiguous())
            reg_outs.append(self.reg(f).permute(0,2,3,1).contiguous())
        return cls_outs, reg_outs


## A.0 Chuẩn bị chung (ảnh + GT + tool vẽ)

In [40]:
import os, cv2, numpy as np, torch
from PIL import Image, ImageDraw

root_dir = "D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/"
IMG_PATH = root_dir + 'input/street.jpg'

if not os.path.exists(IMG_PATH):
    img = Image.new('RGB', (960, 640), (235, 240, 245))
    d = ImageDraw.Draw(img)
    for (x,y,w,h) in [(120,300,120,80),(420,280,90,70),(700,320,160,100)]:
        d.rectangle([x,y,x+w,y+h], outline=(60,60,60), width=4)
    d.text((20,20), "Dummy Street-like Scene", fill=(10,10,10))
    img.save(IMG_PATH)

# GT boxes (xyxy) — ví dụ tay (bạn thay bằng GT thật nếu có)
GT = np.array([
    [90, 780, 210, 965],  # object 1
    [500, 760, 615, 860],  # object 2
    [1100, 750, 1230, 875],  # object 3
], dtype=np.float32)

def draw_boxes(path, boxes, color=(0,255,0), labels=None, scores=None, out='runs_demo/out.jpg', th=0.0):
    os.makedirs(root_dir + 'runs_demo', exist_ok=True)
    im = cv2.imread(path)
    for i,b in enumerate(boxes):
        if scores is not None and scores[i] < th: 
            continue
        x1,y1,x2,y2 = map(int, b)
        cv2.rectangle(im,(x1,y1),(x2,y2),color,2)
        if labels is not None or scores is not None:
            txt=''
            if labels is not None: txt += str(labels[i])
            if scores is not None: txt += f" {scores[i]:.2f}"
            if txt: cv2.putText(im, txt, (x1,max(0,y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    cv2.imwrite(out, im)
    print('Saved:', out)

### A.1 IoU & Encode/Decode (SSD-style)
**Mục tiêu**

* Tính IoU giữa GT và anchors gần nhất.

* Thử encode GT → offsets theo anchor; rồi decode ngược lại để kiểm tra sai số.

In [41]:
# anchors demo (grid đơn giản)
im = cv2.imread(IMG_PATH)
H, W, chal_ = im.shape

feat_h, feat_w = 8, 10  # stride ~32
stride_y, stride_x = H/feat_h, W/feat_w
scales = [0.15, 0.25]; ratios=[1.0, 2.0, 0.5]
anchors=[]

for y in range(feat_h):
    for x in range(feat_w):
        cy=(y+0.5)*stride_y; cx=(x+0.5)*stride_x
        for s in scales:
            for r in ratios:
                w=W*s*(r**0.5); h=H*s/(r**0.5)
                anchors.append([cx-0.5*w, cy-0.5*h, cx+0.5*w, cy+0.5*h])
anchors = torch.tensor(anchors, dtype=torch.float32)

gt = torch.tensor(GT, dtype=torch.float32)
IOU = iou(gt, anchors)                       # [n_gt, n_anchor]

print(IOU.shape)

best_idx = IOU.argmax(dim=1)                 # anchor tốt nhất cho mỗi GT
best_anchors = anchors[best_idx]

# SSD encode/decode
def ssd_encode(gt_xyxy, anc_xyxy, variances=(0.1,0.2)):
    v1,v2 = variances
    gt_xywh = box_xyxy_to_xywh(gt_xyxy); anc_xywh = box_xyxy_to_xywh(anc_xyxy)
    t_xy = (gt_xywh[:,:2]-anc_xywh[:,:2]) / anc_xywh[:,2:]
    t_wh = torch.log(gt_xywh[:,2:]/anc_xywh[:,2:])
    return torch.cat([t_xy/v1, t_wh/v2], dim=-1)

def ssd_decode(t, anc_xyxy, variances=(0.1,0.2)):
    v1,v2 = variances
    anc_xywh = box_xyxy_to_xywh(anc_xyxy)
    cxcy = t[:,:2]*v1*anc_xywh[:,2:] + anc_xywh[:,:2]
    wh   = torch.exp(t[:,2:]*v2)*anc_xywh[:,2:]
    return box_xywh_to_xyxy(torch.cat([cxcy,wh], dim=-1))

t = ssd_encode(gt, best_anchors)
gt_rec = ssd_decode(t, best_anchors)
print('Max L1 decode error:', (gt_rec-gt).abs().max().item())

# Vẽ GT (xanh) & anchor tốt nhất (vàng) & decode (lục)
draw_boxes(IMG_PATH, anchors.numpy(), color=(0,0,255), out=root_dir +'runs_demo/a1_anchors.jpg')

draw_boxes(IMG_PATH, GT, color=(0,255,0), out=root_dir +'runs_demo/a1_gt.jpg')
draw_boxes(IMG_PATH, best_anchors.numpy(), color=(0,255,255), out=root_dir +'runs_demo/a1_best_anchors.jpg')
draw_boxes(IMG_PATH, gt_rec.numpy(), color=(255,0,255), out=root_dir +'runs_demo/a1_decoded.jpg')

torch.Size([3, 480])
Max L1 decode error: 0.0
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a1_anchors.jpg
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a1_gt.jpg
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a1_best_anchors.jpg
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a1_decoded.jpg


## A.2 NMS vs Soft-NMS trên cùng ảnh (dùng boxes giả lập)

In [48]:
# boxes “raw” bị chồng lấn + confidence
boxes = torch.tensor([
    [93, 781, 212, 960],  # gần GT1
    [1080,755, 1200,869],  # trùng mạnh
    [503, 758, 611, 862],  # gần GT2
    [1097, 752, 1228, 873],  # gần GT3
], dtype=torch.float32)

scores = torch.tensor([0.88, 0.81, 0.92, 0.84], dtype=torch.float32)

def iou_one(a, b):
    tl = torch.max(a[:2], b[:2]); br = torch.min(a[2:], b[2:])
    wh = (br - tl).clamp(min=0); inter = wh[0]*wh[1]
    area1=(a[2]-a[0])*(a[3]-a[1]); area2=(b[2]-b[0])*(b[3]-b[1])
    return inter/(area1+area2-inter+1e-7)

def nms(boxes, scores, th=0.5):
    keep=[]; idx = scores.argsort(descending=True)
    while idx.numel()>0:
        i = idx[0]; keep.append(i.item())
        if idx.numel()==1: break
        ov = torch.stack([iou_one(boxes[i], boxes[j]) for j in idx[1:]])
        idx = idx[1:][ov<=th]
    return keep

def soft_nms(boxes, scores, th=0.5, sigma=0.5, score_th=0.001):
    idx = scores.argsort(descending=True)
    b = boxes.clone(); s = scores.clone(); keep=[]
    while idx.numel()>0:
        i = idx[0]; keep.append(i.item())
        if idx.numel()==1: break
        others = idx[1:]
        ov = torch.stack([iou_one(b[i], b[j]) for j in others])
        s[others] = s[others]*torch.exp(-(ov**2)/sigma)
        idx = others[s[others]>score_th]
        idx = idx[s[idx].argsort(descending=True)]
    return keep

k1 = nms(boxes, scores, 0.5)
k2 = soft_nms(boxes, scores, 0.5, 0.5, 0.05)
print('NMS keep:', k1, '| Soft-NMS keep:', k2)

draw_boxes(IMG_PATH, boxes.numpy(), scores=scores.numpy(), out=root_dir +'runs_demo/a2_raw.jpg')
draw_boxes(IMG_PATH, boxes.numpy()[k1], scores=scores.numpy()[k1], color=(0,255,0), out=root_dir +'runs_demo/a2_nms.jpg')
draw_boxes(IMG_PATH, boxes.numpy()[k2], scores=scores.numpy()[k2], color=(0,0,255), out=root_dir +'runs_demo/a2_softnms.jpg')


NMS keep: [2, 0, 3] | Soft-NMS keep: [2, 0, 3, 1]
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a2_raw.jpg
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a2_nms.jpg
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a2_softnms.jpg


## A.3 Anchors / Grid YOLO (trực quan hóa tâm lưới + anchor khung)

In [51]:
im = cv2.imread(IMG_PATH)
H, W, chal_ = im.shape

feat_h, feat_w, stride = 40, 60, 16  # ví dụ P3 stride 16
yv, xv = torch.meshgrid(torch.arange(feat_h), torch.arange(feat_w), indexing='ij')
grid = torch.stack((xv, yv), 2).view(-1,2).float()
centers = (grid + 0.5) * stride  # (N,2) pixel

# Vẽ 200 tâm đầu tiên
im = cv2.imread(IMG_PATH)
for cx,cy in centers[:]:
    cv2.circle(im, (int(cx),int(cy)), 1, (0,0,255), -1)

# Vẽ 1-2 anchor rectangles quanh một tâm chọn
cx,cy = centers[800]  # tùy ô
for w,h in [(40,40),(80,40),(40,80)]:
    x1,y1 = int(cx - w/2), int(cy - h/2)
    x2,y2 = int(cx + w/2), int(cy + h/2)
    cv2.rectangle(im,(x1,y1),(x2,y2),(0,255,255),1)

#os.makedirs('runs_demo', exist_ok=True)
cv2.imwrite(root_dir + 'runs_demo/a3_grid_anchors.jpg', im)
print('Saved: runs_demo/a3_grid_anchors.jpg')

Saved: runs_demo/a3_grid_anchors.jpg


## A.4 FPN: trực quan “độ kích hoạt” (heatmap) trên các mức P3/P4/P5

In [56]:
# ==== A.4 (FIXED) FPN: trực quan “độ kích hoạt” P3/P4/P5 ====
import cv2, numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# 1) Đọc ảnh an toàn (tránh stride âm)

img_bgr = cv2.imread(IMG_PATH)
assert img_bgr is not None, f"Không đọc được ảnh: {IMG_PATH}"
img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# (Tuỳ chọn) pad ảnh để H, W chia hết cho 32 giúp downsample ổn định hơn
def pad_to_divisible(im, div=32):
    h, w = im.shape[:2]
    nh = (h + div - 1) // div * div
    nw = (w + div - 1) // div * div
    if (nh, nw) == (h, w): 
        return im
    pad_b = nh - h
    pad_r = nw - w
    # Pad phía dưới/phải để không lệch nội dung
    return cv2.copyMakeBorder(im, 0, pad_b, 0, pad_r, borderType=cv2.BORDER_REPLICATE)

img = pad_to_divisible(img, 32)
t = torch.from_numpy(img).permute(2,0,1).float()/255.0
t = t.unsqueeze(0)

# 2) Backbone nhẹ để demo
base = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT).features.eval()

# Lấy vài mốc làm "C3/C4/C5" (chỉ để minh hoạ; MobileNet không có C-stages chuẩn như ResNet)
# Bạn có thể đổi danh sách taps tuỳ phiên bản torchvision
taps = [3, 6, 12]  # các chỉ số layer trong base.features
def extract_c3c4c5(x):
    feats = {}
    y = x
    for i, m in enumerate(base):
        y = m(y)
        if i in taps:
            feats[i] = y
    # đảm bảo đủ 3 mức
    assert all(k in feats for k in taps), f"Không lấy đủ taps {taps}, có {list(feats.keys())}"
    return feats[taps[0]], feats[taps[1]], feats[taps[2]]

with torch.no_grad():
    C3, C4, C5 = extract_c3c4c5(t)

class SimpleFPN(nn.Module):
    def __init__(self, c3, c4, c5, out=128):
        super().__init__()
        self.l3 = nn.Conv2d(c3, out, 1)
        self.l4 = nn.Conv2d(c4, out, 1)
        self.l5 = nn.Conv2d(c5, out, 1)
        self.s3 = nn.Conv2d(out, out, 3, padding=1)  # <-- dùng padding, không phải p=
        self.s4 = nn.Conv2d(out, out, 3, padding=1)

    def forward(self, C3, C4, C5):
        P5 = self.l5(C5)
        #  UpSample theo kích thước chuẩn của nhánh lateral để tránh lệch 1 pixel
        P4 = self.l4(C4) + F.interpolate(P5, size=C4.shape[-2:], mode='nearest')
        P3 = self.l3(C3) + F.interpolate(P4, size=C3.shape[-2:], mode='nearest')
        P4 = self.s4(P4)
        P3 = self.s3(P3)
        return P3, P4, P5

with torch.no_grad():
    fpn = SimpleFPN(C3.shape[1], C4.shape[1], C5.shape[1], out=64).eval()
    P3, P4, P5 = fpn(C3, C4, C5)

# 3) Vẽ heatmap
def heat_to_img(h, base_img_bgr, out_path):
    # h: [1,C,H,W] -> trung bình kênh -> normalize -> resize -> overlay
    hm = h.abs().mean(1, keepdim=True)  # [1,1,H,W]
    hm = (hm - hm.min()) / (hm.max() - hm.min() + 1e-7)
    hm = hm[0,0].cpu().numpy()
    H, W = base_img_bgr.shape[:2]
    hm = cv2.resize(hm, (W, H))
    heat = (hm*255).astype(np.uint8)
    heat = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    blend = cv2.addWeighted(base_img_bgr, 0.6, heat, 0.4, 0)
    cv2.imwrite(out_path, blend)
    print("Saved:", out_path)

# Chú ý: base_img_bgr cần trùng kích thước với ảnh đã pad
base_img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
#import os
#os.makedirs('runs_demo', exist_ok=True)
heat_to_img(P3, base_img_bgr, root_dir +'runs_demo/a4_p3.jpg')
heat_to_img(P4, base_img_bgr, root_dir +'runs_demo/a4_p4.jpg')
heat_to_img(P5, base_img_bgr, root_dir +'runs_demo/a4_p5.jpg')

Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a4_p3.jpg
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a4_p4.jpg
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a4_p5.jpg


## A.5 “Head” SSD: kiểm tra hình dạng đầu ra (cls/reg) & sanity box-regression

In [59]:
import torch.nn as nn

class SSDHeadDemo(nn.Module):
    def __init__(self, in_ch, num_anchors, num_classes):
        super().__init__()
        self.cls = nn.Conv2d(in_ch, num_anchors*num_classes, 3, padding=1)
        self.reg = nn.Conv2d(in_ch, num_anchors*4, 3, padding=1)
    def forward(self, f):
        B,C,H,W = f.shape
        cls = self.cls(f).permute(0,2,3,1).contiguous().view(B, -1, num_classes)
        reg = self.reg(f).permute(0,2,3,1).contiguous().view(B, -1, 4)
        return cls, reg

num_classes = 4+1   # background + 4 lớp ví dụ
num_anchors = 6     # ví dụ 6 anchor/mỗi ô
fake_feat = torch.randn(1, 64, 40, 60)  # giống P3 ở trên
head = SSDHeadDemo(64, num_anchors, num_classes)
cls_out, reg_out = head(fake_feat)
print('cls_out:', cls_out.shape, 'reg_out:', reg_out.shape)  # -> (1, H*W*num_anchors, num_classes) & (1, *, 4)

# Sanity: thử reg_out + anchors -> decode ra boxes, rồi vẽ thử
# Tạo anchors tương ứng với (H,W) và stride ~16 như A.3
#H,W = 640,960; feat_h,feat_w,stride = 40,60,16

im = cv2.imread(IMG_PATH)
H, W, chal_ = im.shape
feat_h,feat_w,stride = 40,60,16

yv,xv = torch.meshgrid(torch.arange(feat_h), torch.arange(feat_w), indexing='ij')
centers = (torch.stack((xv,yv),2).view(-1,2).float()+0.5)*stride
anchors=[]
for (w,h) in [(40,40),(80,40),(40,80),(64,64),(96,48),(48,96)]:
    cxcywh = torch.cat([centers, torch.full_like(centers, w), torch.full_like(centers, h)], dim=1)
    # nhưng cần (cx,cy,w,h) — chuyển sang xyxy:
    for i in range(cxcywh.shape[0]):
        cx,cy= cxcywh[i,0], cxcywh[i,1]
        anchors.append([cx-0.5*w, cy-0.5*h, cx+0.5*w, cy+0.5*h])
anchors = torch.tensor(anchors, dtype=torch.float32)

# Lấy N điểm reg bất kỳ & giả lập decode (dùng ssd_decode ở A.1)
def ssd_decode_xyxy(t, anc_xyxy, v=(0.1,0.2)):
    v1,v2=v
    def xyxy2xywh(b):
        x1,y1,x2,y2 = b.unbind(-1); w=x2-x1; h=y2-y1
        cx=x1+0.5*w; cy=y1+0.5*h
        return torch.stack([cx,cy,w,h],-1)
    def xywh2xyxy(b):
        cx,cy,w,h = b.unbind(-1)
        x1=cx-0.5*w; y1=cy-0.5*h; x2=cx+0.5*w; y2=cy+0.5*h
        return torch.stack([x1,y1,x2,y2],-1)
    anc = xyxy2xywh(anc_xyxy)
    cxcy = t[:,:2]*v1*anc[:,2:] + anc[:,:2]
    wh   = torch.exp(t[:,2:]*v2)*anc[:,2:]
    return xywh2xyxy(torch.cat([cxcy,wh],-1))

sel = torch.randint(0, reg_out.shape[1], (200,))  # 200 anchor ngẫu nhiên
reg_sel = reg_out[0, sel] * 0.0  # đặt offset=0 để decode ~ về chính anchor (sanity)
boxes_dec = ssd_decode_xyxy(reg_sel, anchors[sel])
#draw_boxes(IMG_PATH, boxes_dec.numpy(), color=(255,0,0), out=root_dir +'runs_demo/a5_head_sanity.jpg')
draw_boxes(IMG_PATH, boxes_dec.detach().numpy(), color=(255,0,0), out=root_dir +'runs_demo/a5_head_sanity.jpg')

cls_out: torch.Size([1, 14400, 5]) reg_out: torch.Size([1, 14400, 4])
Saved: D:/Teach_n_Train/Advanced_Lessons_CV/LABS/Object_Detection/Lab05/runs_demo/a5_head_sanity.jpg
